In [1]:
%%writefile deviceProperties.cu
#include <stdio.h>
#include <cuda_runtime.h>

int getCoresPerSM(int major, int minor) {
    // Defines cores per SM based on architecture generation
    switch (major) {
 case 2: // Fermi
            return (minor == 1) ? 48 : 32;
        case 3: // Kepler
            return 192;
        case 5: // Maxwell
            return 128;
        case 6: // Pascal
            if (minor == 1 || minor == 2) return 128;
            if (minor == 0) return 64;
            return 128; // Default fallback for Pascal
        case 7: // Volta (7.0), Turing (7.5)
            return 64;
        case 8: // Ampere (8.0, 8.6, 8.7), Ada Lovelace (8.9)
            if (minor == 0) return 64;
            if (minor == 6 || minor == 9) return 128;
            return 64; // Default fallback for Ampere variants
        case 9: // Hopper (9.0), Blackwell (9.5)
            return 128;
        default:
            return 128; // Standard fallback for future architectures
    }
}

int main(){
    int deviceCount = 0;
    cudaError_t error = cudaGetDeviceCount(&deviceCount);
    if (error != cudaSuccess){
        printf("Failed to get Device Count\n");
        return 1;
    }
    printf("Found %d device(s)\n",deviceCount);

    // loop through each device and print their properties
    for (int i=0;i<deviceCount;i++){
        cudaDeviceProp prop;
        error = cudaGetDeviceProperties(&prop, i);

        if (error != cudaSuccess){
            printf("Cannot get properties of device %d\n",i);
            return 1;
        }

        // print properties
        printf("Device (%d): %s\n",i,prop.name);
        printf("    Compute capability: %d.%d\n",prop.major,prop.minor);
        printf("    Number of SMs: %d\n",prop.multiProcessorCount);
        printf("    Cores per SM: %d\n",getCoresPerSM(prop.major, prop.minor));
        printf("    Max blocks per SM: %d\n",prop.maxBlocksPerMultiProcessor);
        printf("    Max number of threads per SM: %d\n",prop.maxThreadsPerMultiProcessor);
        printf("    Max number of threads per block: %d\n",prop.maxThreadsPerBlock);
        printf("    Max dimension of grid: (%d, %d, %d)\n",prop.maxGridSize[0],prop.maxGridSize[1],prop.maxGridSize[2]);
        printf("    Max dimension of block: (%d,%d,%d)\n",prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
        printf("    Registers per block: %d\n",prop.regsPerBlock);
        printf("    Registers per SM: %d\n",prop.regsPerMultiprocessor);
        printf("    Shared memory per block: %lu KB\n",prop.sharedMemPerBlock/(1024));
        printf("    Shared memory per SM: %lu KB\n",prop.sharedMemPerMultiprocessor/(1024));
        printf("    Global memory on device: %lu MB\n",prop.totalGlobalMem/(1024*1024));
        printf("    Constant memory on device: %lu bytes\n",prop.totalConstMem);
        printf("    Warp size: %d threads\n",prop.warpSize);
        printf("    L2 cache size: %d bytes\n", prop.l2CacheSize);
        printf("    Max persisting L2 cache size: %d bytes\n", prop.persistingL2CacheMaxSize);
        printf("    Max pitch allowed by memory copies: %lu MB\n",prop.memPitch/(1024*1024));
        printf("    Memory bus width: %d bits\n",prop.memoryBusWidth);
    }
}

Writing deviceProperties.cu


In [2]:
!nvcc deviceProperties.cu -o deviceProperties

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./deviceProperties

Found 1 device(s)
Device (0): Tesla T4
    Compute capability: 7.5
    Number of SMs: 40
    Cores per SM: 64
    Max blocks per SM: 16
    Max number of threads per SM: 1024
    Max number of threads per block: 1024
    Max dimension of grid: (2147483647, 65535, 65535)
    Max dimension of block: (1024,1024,64)
    Registers per block: 65536
    Registers per SM: 65536
    Shared memory per block: 48 KB
    Shared memory per SM: 64 KB
    Global memory on device: 14912 MB
    Constant memory on device: 65536 bytes
    Warp size: 32 threads
    L2 cache size: 4194304 bytes
    Max persisting L2 cache size: 0 bytes
    Max pitch allowed by memory copies: 2047 MB
    Memory bus width: 256 bits


In [4]:
%%writefile global_sync.cu
#include <stdio.h>
#include <cuda_runtime.h>

// volatile so that other threads can see the updates to 'lock'
__device__ volatile int lock = 0;

__global__ void K(){
    // increment lock once for every block
    if (threadIdx.x == 0)
        atomicAdd((int*)&lock,1);

    // barrier
    if (threadIdx.x == 0)
        while (lock != 32);
    __syncthreads();
    // barrier

    if (threadIdx.x == 0){
        printf("Block Id: %d, lock = %d\n", blockIdx.x, lock);
    }
}

int main(){
    int numBlocks = 32;
    int threadsPerBlock = 512;
    K<<<numBlocks,threadsPerBlock>>>();
    cudaDeviceSynchronize();
    printf("Completed kernel\n");
}

Writing global_sync.cu


In [5]:
!nvcc global_sync.cu -o global_sync

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./global_sync

Block Id: 21, lock = 32
Block Id: 24, lock = 32
Block Id: 11, lock = 32
Block Id: 14, lock = 32
Block Id: 1, lock = 32
Block Id: 4, lock = 32
Block Id: 31, lock = 32
Block Id: 29, lock = 32
Block Id: 26, lock = 32
Block Id: 19, lock = 32
Block Id: 16, lock = 32
Block Id: 9, lock = 32
Block Id: 6, lock = 32
Block Id: 22, lock = 32
Block Id: 12, lock = 32
Block Id: 2, lock = 32
Block Id: 20, lock = 32
Block Id: 30, lock = 32
Block Id: 0, lock = 32
Block Id: 10, lock = 32
Block Id: 27, lock = 32
Block Id: 17, lock = 32
Block Id: 7, lock = 32
Block Id: 23, lock = 32
Block Id: 25, lock = 32
Block Id: 13, lock = 32
Block Id: 15, lock = 32
Block Id: 3, lock = 32
Block Id: 5, lock = 32
Block Id: 28, lock = 32
Block Id: 18, lock = 32
Block Id: 8, lock = 32
Completed kernel
